# Phase 3: Stochastic Multi-Expert (SME) Training
**Paper Eq. 6-11:** Gumbel-Softmax gating over K compressed experts.

### Key design (fixed to match paper):
- Gate depends on **expert outputs** (Eq. 10), not raw input
- Stochastic selection via Gumbel-Softmax with temperature annealing (Eq. 11)
- Weighted aggregation: `y = (1/K) * sum(alpha_j * y_j)` (Eq. 9)
- Adversarial training of gating network with PGD attacks

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import torchattacks
import os, time
from tqdm import tqdm
from utils import load_checkpoint, save_checkpoint, evaluate, count_zero_params
from ensemble_utils import SME_Ensemble

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cuda


In [2]:
# Data loading
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
])
transform_test = transforms.Compose([transforms.ToTensor()])

trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform_train)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=128, shuffle=True, num_workers=2)

testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform_test)
testloader = torch.utils.data.DataLoader(testset, batch_size=100, shuffle=False, num_workers=2)

# Attack parameters
eps, alpha, steps = 8/255, 2/255, 20

In [3]:
# Load the 3 ExPSO-compressed experts from Phase 2
expert_ratios = [30, 50, 70]
experts = []

for r in expert_ratios:
    model = torchvision.models.resnet18(weights=None, num_classes=10).to(device)
    model = load_checkpoint(model, f'checkpoints/expso_expert_{r}.pth')
    model.eval()
    experts.append(model)
    clean = evaluate(model, testloader)
    robust = evaluate(model, testloader, atk=torchattacks.PGD(model, eps=eps, alpha=alpha, steps=20))
    print(f'Expert {r}%: Clean={clean:.2f}%, Robust={robust:.2f}%')

# Create SME Ensemble (fixed: gating depends on expert outputs, Eq. 10-11)
sme_model = SME_Ensemble(experts, num_classes=10, train_experts=False).to(device)
print(f'\nSME Ensemble: {sme_model.k} experts, gate input: {sme_model.k * sme_model.num_classes + sme_model.k * sme_model.feature_dim} dims')
print(f'Gate MLP: {sme_model.gate_W}')

# Verify only gating params are trainable
trainable = sum(p.numel() for p in sme_model.parameters() if p.requires_grad)
total = sum(p.numel() for p in sme_model.parameters())
print(f'Trainable params: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)')

Expert 30%: Clean=71.46%, Robust=39.26%
Expert 50%: Clean=69.05%, Robust=38.97%
Expert 70%: Clean=65.99%, Robust=37.37%

SME Ensemble: 3 experts, gate input: 1566 dims
Gate MLP: Sequential(
  (0): Linear(in_features=1566, out_features=512, bias=True)
  (1): ReLU()
  (2): Dropout(p=0.1, inplace=False)
  (3): Linear(in_features=512, out_features=3, bias=True)
)
Trainable params: 803,843 / 34,348,769 (2.34%)


In [4]:
# ==========================================================================
# SME GATING NETWORK TRAINING
# ==========================================================================
TRAIN_EPOCHS = 80
INITIAL_TAU = 2.0
FINAL_TAU = 0.1
# Only train gating parameters (experts are frozen)
optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, sme_model.parameters()),
    lr=1e-3
)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=TRAIN_EPOCHS)
criterion = nn.CrossEntropyLoss()
# Adversarial training for the router
train_atk = torchattacks.PGD(sme_model, eps=eps, alpha=alpha, steps=10)
def train_sme_one_epoch(model, loader, optimizer, criterion, epoch, tau):
    model.train()
    running_loss, correct, total = 0, 0, 0
    pbar = tqdm(loader, desc=f'Epoch {epoch} [Tau={tau:.2f}]')
    for inputs, targets in pbar:
        inputs, targets = inputs.to(device), targets.to(device)
        # Generate adversarial examples
        adv_inputs = train_atk(inputs, targets)
        optimizer.zero_grad()
        outputs = model(adv_inputs, tau=tau)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += targets.size(0)
        correct += predicted.eq(targets).sum().item()
        pbar.set_postfix({'Loss': f'{running_loss/len(loader):.3f}',
                          'Acc': f'{100.*correct/total:.2f}%'})
# Main training loop
best_robust = 0
patience_counter = 0
patience = 3
import math
print('\nStarting SME Gating Network Training...')
for epoch in range(1, TRAIN_EPOCHS + 1):
    # Exponential temperature cooling
    decay_rate = -math.log(FINAL_TAU / INITIAL_TAU) / max(1, TRAIN_EPOCHS - 1)
    current_tau = INITIAL_TAU * math.exp(-decay_rate * (epoch - 1))
    train_sme_one_epoch(sme_model, trainloader, optimizer, criterion, epoch, current_tau)
    scheduler.step()
    if epoch % 5 == 0 or epoch == TRAIN_EPOCHS:
        sme_model.eval()
        clean = evaluate(sme_model, testloader)
        robust = evaluate(sme_model, testloader,
            atk=torchattacks.PGD(sme_model, eps=eps, alpha=alpha, steps=20))
        print(f'-> Eval: Clean={clean:.2f}%, Robust(PGD-20)={robust:.2f}%')
        if robust > best_robust:
            best_robust = robust
            patience_counter = 0
            save_checkpoint(sme_model, 'checkpoints/sme_ensemble_final_v2.pth')
            print(f'*** New Best SME Model Saved! ***')
        else:
            patience_counter += 1
            print(f'   [Early Stopping] No improvement. Patience: {patience_counter}/{patience}')
            if patience_counter >= patience:
                print('*** Early Stopping Triggered! ***')
                break



Starting SME Gating Network Training...


Epoch 5 [Tau=1.72]: 100%|██████████| 391/391 [03:02<00:00,  2.14it/s, Loss=1.636, Acc=56.32%]


-> Eval: Clean=71.56%, Robust(PGD-20)=47.74%
*** New Best SME Model Saved! ***


Epoch 10 [Tau=1.42]: 100%|██████████| 391/391 [02:59<00:00,  2.17it/s, Loss=1.636, Acc=56.33%]


-> Eval: Clean=71.30%, Robust(PGD-20)=47.80%
*** New Best SME Model Saved! ***


...

-> Eval: Clean=71.52%, Robust(PGD-20)=47.30%
   [Early Stopping] No improvement. Patience: 1/3


Epoch 30 [Tau=0.67]: 100%|██████████| 391/391 [02:57<00:00,  2.20it/s, Loss=1.635, Acc=56.17%]


-> Eval: Clean=71.34%, Robust(PGD-20)=47.90%
   [Early Stopping] No improvement. Patience: 2/3


Epoch 35 [Tau=0.55]: 100%|██████████| 391/391 [03:00<00:00,  2.17it/s, Loss=1.635, Acc=56.16%]


-> Eval: Clean=71.52%, Robust(PGD-20)=47.60%
   [Early Stopping] No improvement. Patience: 3/3
*** Early Stopping Triggered! ***


In [5]:
# ==========================================================================
# PHASE 3: FINAL EVALUATION
# ==========================================================================
sme_model.load_state_dict(torch.load('checkpoints/sme_ensemble_final_v2.pth'))
sme_model.eval()

print('\n' + '='*50)
print('PHASE 3: SME FINAL EVALUATION')
print('='*50)

metrics = {
    'Clean': None,
    'PGD-20': torchattacks.PGD(sme_model, eps=eps, alpha=alpha, steps=20),
    'PGD-100': torchattacks.PGD(sme_model, eps=eps, alpha=alpha, steps=100),
    'FGSM': torchattacks.FGSM(sme_model, eps=eps)
}

for name, atk in metrics.items():
    acc = evaluate(sme_model, testloader, atk=atk)
    print(f'{name:<10}: {acc:.2f}%')


PHASE 3: SME FINAL EVALUATION
Clean     : 71.29%
PGD-20    : 47.67%
PGD-100   : 47.27%
FGSM      : 51.25%
